### Create a tf-idf-based classificator model for the bank77 dataset

In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.ensemble import GradientBoostingClassifier
from utils.utils import preprocessing

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\vojta\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\vojta\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\vojta\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [3]:
import os
import json
import pandas as pd

def extract_input_texts_from_folder(folder_path):
    records = []

    for filename in os.listdir(folder_path):
        if filename.endswith(".jsonl"):
            file_path = os.path.join(folder_path, filename)
            with open(file_path, 'r', encoding='utf-8') as f:
                for line in f:
                    try:
                        record = json.loads(line)
                        custom_id = record.get("custom_id")
                        messages = record.get("body", {}).get("messages", [])
                        for message in messages:
                            if message.get("role") == "user":
                                user_content = message.get("content", "")
                                nested_json = json.loads(user_content)
                                input_text = nested_json.get("input_text")
                                if input_text:
                                    records.append({
                                        "custom_id": custom_id,
                                        "input_text": input_text
                                    })
                    except Exception as e:
                        print(f"Error in file {filename}, skipping line: {e}")

    return pd.DataFrame(records)


In [4]:
from datasets import load_dataset

data_load = load_dataset("banking77")

C:\Users\vojta\miniconda3\envs\llm-features\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
df_train = pd.DataFrame(data_load['train'])
df_test = pd.DataFrame(data_load['test'])

In [6]:
df_test['text'] = df_test.text.apply(lambda x: preprocessing(x))
df_train['text'] = df_train.text.apply(lambda x: preprocessing(x))

In [7]:
# # Create a tf-idf matrix
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df_train['text'])
y = df_train['label']

In [8]:
# # Train a classifier
clf = GradientBoostingClassifier(random_state=42)
clf.fit(X, y)

GradientBoostingClassifier(random_state=42)

In [9]:
# Test the classifier
X_test = vectorizer.transform(df_test['text'])
y_test = df_test['label']
y_pred = clf.predict(X_test)

from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.95      0.95      0.95        40
           1       0.91      1.00      0.95        40
           2       0.97      0.97      0.97        40
           3       0.76      0.62      0.68        40
           4       0.94      0.82      0.88        40
           5       0.44      0.78      0.56        40
           6       0.86      0.95      0.90        40
           7       0.83      0.88      0.85        40
           8       0.87      0.82      0.85        40
           9       1.00      0.93      0.96        40
          10       0.69      0.55      0.61        40
          11       0.50      0.80      0.62        40
          12       0.82      0.70      0.76        40
          13       0.90      0.93      0.91        40
          14       0.62      0.33      0.43        40
          15       0.54      0.68      0.60        40
          16       0.60      0.70      0.64        40
          17       0.80    

### Dummy Classifier

In [10]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import recall_score, f1_score, precision_score, accuracy_score
dummy = DummyClassifier(strategy='uniform', random_state=42)
dummy.fit(X, y)
y_pred = dummy.predict(X_test)
display(f"Dummy Accuracy score: {accuracy_score(y_test, y_pred)}")
display(f"Dummy Recall score: {recall_score(y_test, y_pred, average='macro')}")
display(f"Dummy F1 score: {f1_score(y_test, y_pred, average='macro')}")
display(f"Dummy Precision score: {precision_score(y_test, y_pred, average='macro', zero_division=0)}")


'Dummy Accuracy score: 0.012662337662337663'

'Dummy Recall score: 0.012662337662337661'

'Dummy F1 score: 0.012283485327424105'

'Dummy Precision score: 0.012081139883176413'